================================================================
FILE : evaluate.py
WHAT THIS FILE DOES :
  Step 1 - Load trained model + train.csv + test.csv
  Step 2 - Show training data and test data on screen
  Step 3 - Predict on TRAINING data (check training accuracy)
  Step 4 - Predict on TEST data (check REAL accuracy)
  Step 5 - Calculate 7 metrics
  Step 6 - Generate and save 7 plots
  Step 7 - Print full results below

HOW TO RUN :  python evaluate.py
RUN AFTER  :  train.py
================================================================

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    classification_report
)

In [ ]:
os.makedirs("results/plots", exist_ok=True)

In [ ]:
print("=" * 65)
print("        MODEL EVALUATION  —  Step 3 of 4")
print("=" * 65)

================================================================
STEP 1 : LOAD MODEL AND DATA
================================================================

In [ ]:
print("\n[STEP 1]  Loading trained model and data...")

In [ ]:
with open("results/random_forest.pkl",  "rb") as f: rf            = pickle.load(f)
with open("results/feature_names.pkl",  "rb") as f: feature_names = pickle.load(f)

In [ ]:
train_df = pd.read_csv("data/train.csv")
test_df  = pd.read_csv("data/test.csv")

In [ ]:
X_train = train_df.drop(columns=["label"])
y_train = train_df["label"].values

In [ ]:
X_test  = test_df.drop(columns=["label"])
y_test  = test_df["label"].values

In [ ]:
print(f"         Random Forest model loaded")
print(f"         train.csv loaded  :  {train_df.shape[0]} rows")
print(f"         test.csv  loaded  :  {test_df.shape[0]} rows")

================================================================
STEP 2 : SHOW TRAINING DATA AND TEST DATA
================================================================

In [ ]:
print("\n" + "=" * 65)
print("  TRAINING DATA  —  Overview")
print("=" * 65)
print(f"  Rows     :  {len(train_df)}")
print(f"  Columns  :  {train_df.shape[1]}")
print(f"  Benign (0)    :  {(y_train==0).sum()}  ({(y_train==0).sum()/len(y_train)*100:.1f}%)")
print(f"  Malicious (1) :  {(y_train==1).sum()}  ({(y_train==1).sum()/len(y_train)*100:.1f}%)")
print()
cols_show = list(X_train.columns[:6]) + ["label"]
print("  Sample (first 5 rows):")
print(train_df[cols_show].head(5).to_string(index=True))

In [ ]:
print("\n" + "=" * 65)
print("  TEST DATA  —  Overview  (model never saw this during training)")
print("=" * 65)
print(f"  Rows     :  {len(test_df)}")
print(f"  Columns  :  {test_df.shape[1]}")
print(f"  Benign (0)    :  {(y_test==0).sum()}  ({(y_test==0).sum()/len(y_test)*100:.1f}%)")
print(f"  Malicious (1) :  {(y_test==1).sum()}  ({(y_test==1).sum()/len(y_test)*100:.1f}%)")
print()
print("  Sample (first 5 rows):")
print(test_df[cols_show].head(5).to_string(index=True))

================================================================
STEP 3 : PREDICT ON TRAINING DATA
================================================================
Training accuracy shows how well model learned the training data
We expect ~100% — model has seen all these files before

In [ ]:
print("\n[STEP 3]  Predicting on TRAINING data...")

In [ ]:
train_proba = rf.predict_proba(X_train)[:, 1]
train_pred  = rf.predict(X_train)
train_acc   = accuracy_score(y_train, train_pred) * 100
train_correct = (train_pred == y_train).sum()

In [ ]:
print(f"\n  TRAINING RESULTS")
print(f"  Accuracy        :  {train_acc:.2f}%")
print(f"  Correct         :  {train_correct} out of {len(y_train)}")
print(f"  Wrong           :  {len(y_train) - train_correct}")

In [ ]:
print()
print("  Prediction sample (first 10 training files):")
print("  " + "-" * 55)
print(f"  {'#':>4}  {'Actual':>10}  {'Predicted':>10}  {'Score':>8}  {'OK?':>5}")
print("  " + "-" * 55)
for i in range(10):
    actual    = "Malicious" if y_train[i] == 1 else "Benign"
    predicted = "Malicious" if train_pred[i] == 1 else "Benign"
    score     = train_proba[i]
    ok        = "YES" if train_pred[i] == y_train[i] else "NO"
    print(f"  {i+1:>4}  {actual:>10}  {predicted:>10}  {score:>8.4f}  {ok:>5}")
print("  " + "-" * 55)

================================================================
STEP 4 : PREDICT ON TEST DATA
================================================================
THIS IS THE REAL TEST — model has NEVER seen these files before
This simulates real-world use where new unknown files arrive

In [ ]:
print("\n[STEP 4]  Predicting on TEST data  (files model never saw)...")

In [ ]:
test_proba = rf.predict_proba(X_test)[:, 1]
test_pred  = rf.predict(X_test)
test_acc   = accuracy_score(y_test, test_pred) * 100
test_correct = (test_pred == y_test).sum()

In [ ]:
print(f"\n  TEST RESULTS  (real-world accuracy)")
print(f"  Accuracy        :  {test_acc:.2f}%")
print(f"  Correct         :  {test_correct} out of {len(y_test)}")
print(f"  Wrong           :  {len(y_test) - test_correct}")

In [ ]:
print()
print("  Prediction sample (first 10 test files):")
print("  " + "-" * 55)
print(f"  {'#':>4}  {'Actual':>10}  {'Predicted':>10}  {'Score':>8}  {'OK?':>5}")
print("  " + "-" * 55)
for i in range(10):
    actual    = "Malicious" if y_test[i] == 1 else "Benign"
    predicted = "Malicious" if test_pred[i] == 1 else "Benign"
    score     = test_proba[i]
    ok        = "YES" if test_pred[i] == y_test[i] else "NO"
    print(f"  {i+1:>4}  {actual:>10}  {predicted:>10}  {score:>8.4f}  {ok:>5}")
print("  " + "-" * 55)

In [ ]:
# Show wrong predictions
wrong_idx = np.where(test_pred != y_test)[0]
print(f"\n  FILES PREDICTED WRONG  ({len(wrong_idx)} total):")
if len(wrong_idx) > 0:
    print("  " + "-" * 55)
    print(f"  {'#':>4}  {'Actual':>10}  {'Predicted':>10}  {'Score':>8}")
    print("  " + "-" * 55)
    for i in wrong_idx[:10]:
        actual    = "Malicious" if y_test[i] == 1 else "Benign"
        predicted = "Malicious" if test_pred[i] == 1 else "Benign"
        print(f"  {i+1:>4}  {actual:>10}  {predicted:>10}  {test_proba[i]:>8.4f}")
    if len(wrong_idx) > 10:
        print(f"  ... and {len(wrong_idx)-10} more")
    print("  " + "-" * 55)

================================================================
STEP 5 : CALCULATE 7 METRICS
================================================================
All metrics calculated on TEST DATA (real-world performance)

ACCURACY   = correct / total
             "Out of all files, how many did we get right?"

PRECISION  = TP / (TP + FP)
             "Of all files we flagged malicious, how many really were?"
             High precision = fewer false alarms

RECALL     = TP / (TP + FN)
             "Of ALL actual malware, how many did we CATCH?"
             High recall = fewer missed threats
             THIS IS THE MOST IMPORTANT METRIC FOR SECURITY
             Missing malware = attacker gets through!

F1 SCORE   = 2 × (Precision × Recall) / (Precision + Recall)
             Best single number balancing precision and recall

ROC-AUC    = Area under ROC curve
             "How well does model RANK files suspicious to safe?"
             1.0 = perfect,  0.5 = random guessing

PR-AUC     = Area under Precision-Recall curve
             Good metric for imbalanced datasets

MCC        = Matthews Correlation Coefficient
             Most honest single metric, considers all 4 outcomes
             Range: -1 to +1,  +1 = perfect

In [ ]:
print("\n[STEP 5]  Calculating 7 metrics on test data...")

In [ ]:
accuracy  = accuracy_score(y_test, test_pred)
precision = precision_score(y_test, test_pred, zero_division=0)
recall    = recall_score(y_test, test_pred, zero_division=0)
f1        = f1_score(y_test, test_pred, zero_division=0)
roc_auc   = roc_auc_score(y_test, test_proba)
pr_auc    = average_precision_score(y_test, test_proba)
mcc       = matthews_corrcoef(y_test, test_pred)

In [ ]:
# Confusion matrix values
cm = confusion_matrix(y_test, test_pred)
tn, fp, fn, tp = cm.ravel()

In [ ]:
print()
print("  " + "=" * 50)
print("  METRIC RESULTS  (on 4351 unseen test files)")
print("  " + "=" * 50)
print(f"  Accuracy   :  {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"             → Got {test_correct} out of {len(y_test)} files correct")
print()
print(f"  Precision  :  {precision:.4f}  ({precision*100:.2f}%)")
print(f"             → Of all we flagged malicious, {precision*100:.1f}% really were")
print()
print(f"  Recall     :  {recall:.4f}  ({recall*100:.2f}%)  ← MOST IMPORTANT")
print(f"             → Caught {recall*100:.1f}% of ALL actual malware")
print(f"             → Missed only {fn} malware files out of {(y_test==1).sum()}")
print()
print(f"  F1 Score   :  {f1:.4f}  ({f1*100:.2f}%)")
print(f"             → Balance between precision and recall")
print()
print(f"  ROC-AUC    :  {roc_auc:.4f}")
print(f"             → 1.0=perfect  0.5=random guessing")
print()
print(f"  PR-AUC     :  {pr_auc:.4f}")
print(f"             → Precision-recall area under curve")
print()
print(f"  MCC        :  {mcc:.4f}")
print(f"             → Most honest metric  (range -1 to +1,  +1=perfect)")
print()
print("  CONFUSION MATRIX BREAKDOWN")
print(f"    True Negatives  (correctly said Benign)    :  {tn}")
print(f"    True Positives  (correctly caught malware) :  {tp}")
print(f"    False Positives (wrongly flagged safe file):  {fp}")
print(f"    False Negatives (MISSED real malware)      :  {fn}  ← want this LOW")
print("  " + "=" * 50)

In [ ]:
print()
print("  FULL CLASSIFICATION REPORT:")
print()
print(classification_report(y_test, test_pred, target_names=["Benign", "Malicious"]))

In [ ]:
# Save metrics to CSV
metrics_df = pd.DataFrame([{
    "Model"    : "Random Forest",
    "Accuracy" : round(accuracy,  4),
    "Precision": round(precision, 4),
    "Recall"   : round(recall,    4),
    "F1"       : round(f1,        4),
    "ROC-AUC"  : round(roc_auc,   4),
    "PR-AUC"   : round(pr_auc,    4),
    "MCC"      : round(mcc,       4),
    "TP"       : int(tp),
    "TN"       : int(tn),
    "FP"       : int(fp),
    "FN"       : int(fn),
}])
metrics_df.to_csv("results/metrics_summary.csv", index=False)
print("  Metrics saved  →  results/metrics_summary.csv")

================================================================
STEP 6 : GENERATE AND SAVE 7 PLOTS
================================================================

In [ ]:
print("\n[STEP 6]  Generating plots...")

In [ ]:
# ── PLOT 1 : Confusion Matrix ────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap([[tn, fp],[fn, tp]], annot=True, fmt="d", cmap="Blues",
            xticklabels=["Benign","Malicious"],
            yticklabels=["Benign","Malicious"],
            annot_kws={"size":16})
ax.set_xlabel("Predicted Label", fontsize=12)
ax.set_ylabel("Actual Label",    fontsize=12)
ax.set_title("Confusion Matrix — Random Forest", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("results/plots/confusion_matrix.png", dpi=150)
plt.close()
print("  Plot 1 saved  →  results/plots/confusion_matrix.png")

In [ ]:
# ── PLOT 2 : ROC Curve ───────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_test, test_proba)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(fpr, tpr, color="steelblue", lw=2, label=f"Random Forest  (AUC = {roc_auc:.4f})")
ax.plot([0,1],[0,1], "k--", alpha=0.4, label="Random Guessing  (AUC = 0.5)")
ax.fill_between(fpr, tpr, alpha=0.1, color="steelblue")
ax.set_xlabel("False Positive Rate  (false alarms)", fontsize=11)
ax.set_ylabel("True Positive Rate  (malware caught)", fontsize=11)
ax.set_title("ROC Curve", fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/plots/roc_curve.png", dpi=150)
plt.close()
print("  Plot 2 saved  →  results/plots/roc_curve.png")

In [ ]:
# ── PLOT 3 : Precision-Recall Curve ─────────────────────────────
p_vals, r_vals, _ = precision_recall_curve(y_test, test_proba)
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(r_vals, p_vals, color="steelblue", lw=2, label=f"Random Forest  (AP = {pr_auc:.4f})")
ax.axhline(y_test.mean(), color="gray", linestyle="--", label=f"Baseline = {y_test.mean():.2f}")
ax.set_xlabel("Recall  (how much malware we catch)",   fontsize=11)
ax.set_ylabel("Precision  (accuracy of our alerts)", fontsize=11)
ax.set_title("Precision-Recall Curve", fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/plots/pr_curve.png", dpi=150)
plt.close()
print("  Plot 3 saved  →  results/plots/pr_curve.png")

In [ ]:
# ── PLOT 4 : Score Distribution ──────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(test_proba[y_test==0], bins=50, alpha=0.6, color="steelblue",
        label="Benign files",   density=True)
ax.hist(test_proba[y_test==1], bins=50, alpha=0.6, color="tomato",
        label="Malicious files", density=True)
ax.axvline(0.5, color="black", linestyle="--", lw=2, label="Decision threshold  (0.5)")
ax.set_xlabel("Malicious Probability Score", fontsize=11)
ax.set_ylabel("Density", fontsize=11)
ax.set_title("Score Distribution — Benign vs Malicious", fontsize=13, fontweight="bold")
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/plots/score_distribution.png", dpi=150)
plt.close()
print("  Plot 4 saved  →  results/plots/score_distribution.png")

In [ ]:
# ── PLOT 5 : Feature Importance ──────────────────────────────────
importances = rf.feature_importances_
top_n       = 20
idx         = np.argsort(importances)[-top_n:]
top_feat    = [feature_names[i] for i in idx]
top_vals    = importances[idx]
fig, ax = plt.subplots(figsize=(9, 7))
colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, top_n))
ax.barh(range(top_n), top_vals, color=colors, edgecolor="black", height=0.7)
ax.set_yticks(range(top_n))
ax.set_yticklabels(top_feat, fontsize=10)
ax.set_xlabel("Importance Score", fontsize=11)
ax.set_title(f"Top {top_n} Most Important Features", fontsize=13, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig("results/plots/feature_importance.png", dpi=150)
plt.close()
print("  Plot 5 saved  →  results/plots/feature_importance.png")

In [ ]:
# ── PLOT 6 : Threshold vs F1 ─────────────────────────────────────
thresholds = np.linspace(0.01, 0.99, 200)
f1_scores  = [f1_score(y_test, (test_proba>=t).astype(int), zero_division=0)
              for t in thresholds]
best_t     = thresholds[np.argmax(f1_scores)]
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, f1_scores, color="steelblue", lw=2)
ax.axvline(best_t, color="tomato", linestyle="--", lw=2,
           label=f"Best threshold = {best_t:.2f}  (F1 = {max(f1_scores):.4f})")
ax.axvline(0.5,    color="gray",   linestyle="--", lw=1, label="Default threshold = 0.5")
ax.set_xlabel("Decision Threshold", fontsize=11)
ax.set_ylabel("F1 Score",           fontsize=11)
ax.set_title("Threshold vs F1 Score", fontsize=13, fontweight="bold")
ax.legend(fontsize=10); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/plots/threshold_vs_f1.png", dpi=150)
plt.close()
print("  Plot 6 saved  →  results/plots/threshold_vs_f1.png")

In [ ]:
# ── PLOT 7 : Train vs Test Accuracy ──────────────────────────────
categories   = ["Training Accuracy", "Test Accuracy"]
values       = [train_acc, test_acc]
colors_bar   = ["steelblue", "tomato"]
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(categories, values, color=colors_bar, edgecolor="black", width=0.4)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
            f"{val:.2f}%", ha="center", fontsize=13, fontweight="bold")
ax.set_ylim(90, 102)
ax.set_ylabel("Accuracy (%)", fontsize=11)
ax.set_title("Training Accuracy vs Test Accuracy", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results/plots/train_vs_test_accuracy.png", dpi=150)
plt.close()
print("  Plot 7 saved  →  results/plots/train_vs_test_accuracy.png")

In [ ]:
# ================================================================
# FINAL SUMMARY — printed below the code
# ================================================================
print("\n" + "=" * 65)
print("  EVALUATION COMPLETE — FULL SUMMARY")
print("=" * 65)
print()
print("  DATASET USED FOR EVALUATION")
print(f"    Training files   :  {len(X_train)}")
print(f"    Test files       :  {len(X_test)}")
print(f"    Features         :  {X_train.shape[1]}")
print()
print("  TRAINING PERFORMANCE")
print(f"    Training Accuracy :  {train_acc:.2f}%")
print(f"    Correct           :  {(train_pred==y_train).sum()} / {len(y_train)}")
print()
print("  TEST PERFORMANCE  (real-world accuracy on unseen files)")
print(f"    Accuracy          :  {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"    Precision         :  {precision:.4f}  ({precision*100:.2f}%)")
print(f"    Recall            :  {recall:.4f}  ({recall*100:.2f}%)  ← catches {recall*100:.1f}% of all malware")
print(f"    F1 Score          :  {f1:.4f}  ({f1*100:.2f}%)")
print(f"    ROC-AUC           :  {roc_auc:.4f}")
print(f"    PR-AUC            :  {pr_auc:.4f}")
print(f"    MCC               :  {mcc:.4f}")
print()
print("  CONFUSION MATRIX")
print(f"    True Positives  (caught malware)     :  {tp}")
print(f"    True Negatives  (correct benign)     :  {tn}")
print(f"    False Positives (wrong alarm)        :  {fp}")
print(f"    False Negatives (missed malware)     :  {fn}  ← most dangerous")
print()
print("  PLOTS SAVED")
plots = [
    "confusion_matrix.png       → TP FP TN FN breakdown",
    "roc_curve.png              → ROC curve  AUC=" + str(round(roc_auc,4)),
    "pr_curve.png               → Precision-Recall curve",
    "score_distribution.png     → Score histogram benign vs malicious",
    "feature_importance.png     → Top 20 most important features",
    "threshold_vs_f1.png        → Best decision threshold",
    "train_vs_test_accuracy.png → Training vs test accuracy bar",
]
for p in plots:
    print(f"    {p}")
print()
print("  results/metrics_summary.csv  ← all metrics saved")
print()
print("=" * 65)
print("  NEXT STEP  →  Run  predict.py")
print("=" * 65)